# Chapter 4: Engineer the context layer

Companion to *Agent Harness Engineering*. Listings 4.1 to 4.4 live in `ahe/ch04.py`.

## Setup

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "ahe").exists():
    root = root.parent            # opened from inside chapter-0N/
sys.path.insert(0, str(root))
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
import os
LIVE = os.environ.get("AHE_LIVE") == "1"
print("Python", sys.version.split()[0], "| live calls:", "on" if LIVE else "off")

## 1. The exit test (Table 4.7)

In [ ]:
from ahe import ch03, ch04
from ahe.ch03 import request, run_task, Budget
run, mid = ch04.table_4_3()
print("naive:", ch04.size(ch04.naive_context(run)), "chars")
for row in ch04.exit_test(mid, ch04.P042_AFTER_3, (2400, 1200, 900, 500)):
    print(row)

## 2. What the model saw at one step

In [ ]:
from pprint import pprint
run = run_task(request("P042", 3), ch04.planner_v2, context=ch04.context_v2)
ctx = [e for e in run.events if e["kind"] == "context"][3]
print(ctx["sha"], ctx["chars"], "chars")
pprint(ctx["bundle"]["pending"])

## 3. Replay the four-context experiment (Table 4.4)

In [ ]:
import json
from ahe import live
arms = (("naive", ch03.context_for), ("assembled", ch04.context_without_terms),
        ("terms", ch04.context_v2), ("historyterms", ch04.context_history_with_terms),
        ("allobs", ch04.context_all_observations))
for pid in ("P042", "P043"):
    for tag, context in arms:
        exp = f"ch04_{pid.lower()}_{tag}_gpt-6-luna"
        assert live.records(exp), f"missing recording {exp}"
        runs, _ = live.replay(exp, [pid] * 5,
                              lambda c: ch03.PLANNER_PROMPT + json.dumps(c, default=str),
                              lambda planner, p, context=context: run_task(
                                  request(p, 3), planner, budget=Budget(steps=16, reads=6),
                                  context=context))
        done = sum(r.state == "completed" for r in runs)
        print(pid, f"{tag:13s} completed {done}/5, steps", [r.count("proposal") for r in runs])

### Live calls

Every live cell below does nothing unless `AHE_LIVE=1` is set before Jupyter starts. Settings come from the environment or from `~/.config/ahe/openrouter.env`:

```
AHE_BASE_URL=https://openrouter.ai/api/v1     # or https://<workspace>/serving-endpoints
AHE_API_KEY=...                                # never commit this file
AHE_MODEL=openai/gpt-6-luna
```

Any OpenAI-compatible endpoint works, including a Databricks workspace (base URL `https://<workspace-host>/serving-endpoints`, a personal access token as the key, and a served model name). `ahe.live.Recorder` replaces `chat` so that each call is appended to `runs/<experiment>.jsonl` with its prompt, reply, token counts, and cost. Use a new experiment name for your own runs so the book's records stay as they are.

In [ ]:
if LIVE:
    live.load_env()
    ch01.chat = live.Recorder("my_ch04_terms")
    r = run_task(request("P042", 3), ch03.live_planner, context=ch04.context_v2,
                 budget=Budget(steps=16, reads=6))
    print(r.state, r.reason, r.count("proposal"), "steps")
else:
    print("Set AHE_LIVE=1 to run the live planner on the assembled context.")

## Exercise solutions

### Exercise 4.1

The run fails with a named reason, because `fake_planner` declares context version 1. An adapter can map `recent` to `observations`, but it cannot fill `observations` honestly once the window has dropped any: version 1 promised every observation. So the adapter below refuses as soon as the bundle is missing one, and the run fails with that reason instead of letting the planner act on a partial history.

In [ ]:
r = run_task(request("P042", 3), ch03.fake_planner, context=ch04.context_v2)
print(r.state, r.reason)

def as_v1(bundle):
    missing = bundle["archive"]["observations"] - len(bundle["recent"])
    if missing:                        # v1 promised every observation
        raise ValueError(f"cannot adapt: {missing} observations are not in the bundle")
    return {"task": bundle["task"], "rules": [x["id"] for x in bundle["rules"]],
            "results": {k: v["status"] for k, v in bundle["results"].items()},
            "observations": bundle["recent"], "steps_left": bundle["steps_left"]}
def adapted(context):
    return ch03.fake_planner(as_v1(context))
adapted.context_schema = 2
r = run_task(request("P042", 3), adapted, context=ch04.context_v2)
print(r.state, r.reason)

### Exercise 4.2

The invariant: no note sentence appears in the bundle unless it is a grounded quotation in `results`. Returning the whole note from `t_read` puts every sentence into observations and so into the window. The reviewer would see the same packet; the log would show the planner being handed text no contract checked.

In [ ]:
import json
from ahe.ch02 import sentences
def no_ungrounded_text(run, ctx):
    quotes = {v["quote"] for v in ctx["results"].values() if v.get("quote")}
    text = json.dumps(ctx["recent"], default=str)
    return not any(s in text for n in ch04.NOTES[run.request["patient"]]
                   for s in sentences(n.text) if s not in quotes and len(s) > 20)
run = run_task(request("P042", 3), ch04.planner_v2, context=ch04.context_v2)
print(all(no_ungrounded_text(run, e["bundle"]) for e in run.events if e["kind"] == "context"))

### Exercise 4.3

Handed the note, the stand-in reader grounds it as `present`, so the reader is not the problem. Lexical search with the listed terms never returns it, and the rule stays `unknown`. A fixture that justifies vector search states that n70 is relevant, that a search for anticoagulants must return it, and that `ground` must still find the quotation and date in the note, because a similarity match is not a reading.

In [ ]:
from datetime import date
from ahe.ch02 import read_note, fake_reader
print(read_note(fake_reader, ch04.NOTES["P044"][0], "anticoagulant", date(2026, 9, 1))[:2])
r = run_task(request("P044", 3), ch04.planner_v2, context=ch04.context_v2)
print(r.hits, r.results["no_anticoag"])